In [2]:
import numpy as np
import matplotlib.pyplot as plt

eta_0 = 0.0      # 1st boundry condition (starting one)
eta_infinity = 10.0   # 2nd boundry condition (The last point)

step_sizes = [0.4, 0.2, 0.1, 0.05, 0.025]

tolerance = 1e-10 

print("Computational domain:", eta_0, "to", eta_infinity)
print("Step sizes:", step_sizes)
print("Shooting tolerance:", tolerance)


Computational domain: 0.0 to 10.0
Step sizes: [0.4, 0.2, 0.1, 0.05, 0.025]
Shooting tolerance: 1e-10


In [3]:
def blasius_equations(eta, Y):       # creating the blasius function

    y1, y2, y3 = Y

    dy1 = y2
    dy2 = y3
    dy3 = -0.5 * y1 * y3

    return np.array([dy1, dy2, dy3])


Y_test = np.array([0.0, 0.0, 0.332])     #Testing if the function works

derivatives = blasius_equations(0.0, Y_test)

print("Initial solution vector:", Y_test)
print("Initial derivatives:", derivatives)



Initial solution vector: [0.    0.    0.332]
Initial derivatives: [ 0.     0.332 -0.   ]


In [4]:
def RK3(eta_0, eta_infinity, Y0, h):     #Defining an RK3 function

    N = int(round((eta_infinity - eta_0) / h))
    eta = np.linspace(eta_0, eta_infinity, N + 1)
    Y = np.zeros((N + 1, 3))
    Y[0] = Y0

    for i in range(N):

        k1 = h * blasius_equations(eta[i], Y[i])

        k2 = h * blasius_equations(
            eta[i] + h/2,
            Y[i] + k1/2
        )

        k3 = h * blasius_equations(
            eta[i] + h,
            Y[i] - k1 + 2*k2
        )

        Y[i+1] = Y[i] + (k1 + 4*k2 + k3)/6

    return eta, Y


Y0_test = np.array([0.0, 0.0, 0.332])     #Testing

eta_test, Y_test = RK3(
    eta_0,
    eta_infinity,
    Y0_test,
    0.1
)

print("Final values using RK3:")
print("f(eta_infinity)   =", Y_test[-1, 0])
print("f'(eta_infinity)  =", Y_test[-1, 1])
print("f''(eta_infinity) =", Y_test[-1, 2])



Final values using RK3:
f(eta_infinity)   = 8.278184867449003
f'(eta_infinity)  = 0.99988649492211
f''(eta_infinity) = 8.238392781356104e-09


In [6]:
def RK4(eta_0, eta_infinity, Y0, h):           # Created an RK4 function

    N = int(round((eta_infinity - eta_0) / h))
    eta = np.linspace(eta_0, eta_infinity, N + 1)
    Y = np.zeros((N + 1, 3))
    Y[0] = Y0

    for i in range(N):

        k1 = h * blasius_equations(eta[i], Y[i])

        k2 = h * blasius_equations(
            eta[i] + h/2,
            Y[i] + k1/2
        )

        k3 = h * blasius_equations(
            eta[i] + h/2,
            Y[i] + k2/2
        )

        k4 = h * blasius_equations(
            eta[i] + h,
            Y[i] + k3
        )

        Y[i+1] = Y[i] + (k1 + 2*k2 + 2*k3 + k4)/6

    return eta, Y

Y0_test = np.array([0.0, 0.0, 0.332])             #Testing RK4 function

eta_test_RK4, Y_test_RK4 = RK4(
    eta_0,
    eta_infinity,
    Y0_test,
    0.1
)

print("Final values using RK4:")
print("f(eta_infinity)   =", Y_test_RK4[-1, 0])
print("f'(eta_infinity)  =", Y_test_RK4[-1, 1])
print("f''(eta_infinity) =", Y_test_RK4[-1, 2])



Final values using RK4:
f(eta_infinity)   = 8.278160196441748
f'(eta_infinity)  = 0.9998848647428862
f''(eta_infinity) = 8.477515748867367e-09


In [9]:
def shooting_method(integration_method, h):   #implementing the shotting method using the bisection method

    # Initial bracket for the shooting parameter
    s_low = 0.3
    s_high = 0.4

    def residual(s):

        Y0 = np.array([0.0, 0.0, s])

        eta, Y = integration_method(
            eta_0, eta_infinity, Y0, h
        )

        return Y[-1, 1] - 1.0

    F_low = residual(s_low)            #residual calculation
    F_high = residual(s_high)

    if F_low * F_high > 0:
        raise ValueError("Bisection interval does not bracket a root.")

    iteration = 0         # Bisection iterations
    max_iterations = 100

    while iteration < max_iterations:

        s_mid = (s_low + s_high) / 2

        F_mid = residual(s_mid)

        if abs(F_mid) < tolerance:
            break

        if F_low * F_mid < 0:

            s_high = s_mid
            F_high = F_mid

        else:

            s_low = s_mid
            F_low = F_mid

        iteration += 1

    return s_mid, F_mid, iteration


# Testing the shooting method

s_RK3, residual_RK3, iterations_RK3 = shooting_method(RK3, 0.1)

s_RK4, residual_RK4, iterations_RK4 = shooting_method(RK4, 0.1)

print("Results using RK3:")
print("Shooting parameter =", s_RK3)
print("Residual =", residual_RK3)
print("Iterations =", iterations_RK3)

print("\nResults using RK4:")
print("Shooting parameter =", s_RK4)
print("Residual =", residual_RK4)
print("Iterations =", iterations_RK4)



Results using RK3:
Shooting parameter = 0.33205653335899116
Residual = -8.416689567525282e-11
Iterations = 28

Results using RK4:
Shooting parameter = 0.3320573456585407
Residual = 9.635670039642719e-11
Iterations = 26


In [10]:
RK3_results = []       # We calculate the shooting parameter using RK3 here

for h in step_sizes:

    s, residual, iterations = shooting_method(RK3, h)

    RK3_results.append([h, s, residual, iterations])


print("Results using RK3:")
print("-" * 65)
print("{:<15} {:<20} {:<20} {:<10}".format(
    "Step Size", "Shooting Parameter", "Residual", "Iterations"
))
print("-" * 65)

for result in RK3_results:

    print("{:<15} {:<20.10f} {:<20.3e} {:<10}".format(
        result[0], result[1], result[2], result[3]
    ))

Results using RK3:
-----------------------------------------------------------------
Step Size       Shooting Parameter   Residual             Iterations
-----------------------------------------------------------------
0.4             0.3319771448         8.997e-11            28        
0.2             0.3320495983         3.405e-11            29        
0.1             0.3320565334         -8.417e-11           28        
0.05            0.3320572476         -3.347e-12           29        
0.025           0.3320573267         -8.327e-11           27        


In [12]:
# implementing RK4 for different step sizes
RK4_solutions = []
RK4_residuals = []
RK4_iterations = []

# Calculate the shooting parameter for each step size
for h in step_sizes:

    s, residual_value, iterations = shooting_method(RK4, h)

    RK4_solutions.append(s)
    RK4_residuals.append(residual_value)
    RK4_iterations.append(iterations)

print("Results using RK4")
print("------------------------------------------------")
print("Step Size       Shooting Parameter       Residual")

for i in range(len(step_sizes)):

    print(
        step_sizes[i], "          ",
        RK4_solutions[i], "          ",
        RK4_residuals[i]
    )

Results using RK4
------------------------------------------------
Step Size       Shooting Parameter       Residual
0.4            0.332058815844357            1.0587086762825493e-11
0.2            0.3320574574172496            -6.693434695392853e-11
0.1            0.3320573456585407            9.635670039642719e-11
0.05            0.33205733774229873            -3.001410231462387e-11
0.025            0.3320573372766375            7.520095657298498e-11
